# Multi-Agent Pricer

In [1]:
import os
import json
import re
import logging
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
import gradio as gr

from agents.agent import Agent
from agents.scanner_agent import ScannerAgent
from agents.frontier_agent import FrontierAgent
from agents.messaging_agent import MessagingAgent
from agents.deals import Deal, Opportunity
from agents.deal_agent_framework import DealAgentFramework

load_dotenv(override=True)
logging.getLogger().setLevel(logging.INFO)

In [2]:
#GET API KEYS
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists")
else:
    print("OpenRouter API Key not set")

OpenRouter API Key exists


In [3]:
#initialize agents
openrouter_url = "https://openrouter.ai/api/v1"

openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

In [5]:

def get_price(s: str) -> float:
    s = (s or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(m.group()) if m else 0.0

def _chat_price(client: OpenAI, model: str, description: str) -> float:
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Estimate the price in USD. Reply with only the number.\n\n{description}"}],
        max_tokens=32,
    )
    return get_price(r.choices[0].message.content or "")

class OpenRouterAgent(Agent):
    name, color = "OpenRouter", Agent.CYAN
    def __init__(self, model_id: str):
        self.client = openrouter_client
        self.model = model_id
    def price(self, description: str) -> float:
        return _chat_price(self.client, self.model, description)

In [6]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")


# Pls note that im using free openrouter models for this instead of the Frontier Agent module due to free limits from openai
openrouter0 = OpenRouterAgent("stepfun/step-3.5-flash:free")
openrouter1 = OpenRouterAgent("openai/gpt-oss-120b:free")
openrouter2 = OpenRouterAgent("google/gemma-3-27b-it:free")
weights = (0.4, 0.3, 0.3)

def ensemble_price(description: str) -> float:
    w = weights
    return w[0]*openrouter0.price(description) + w[1]*openrouter1.price(description) + w[2]*openrouter2.price(description)

In [8]:
class PlanningAgentLLM(Agent):
    name, color, DEAL_THRESHOLD = "Planning (LLM)", Agent.GREEN, 50
    def __init__(self, collection):
        self.scanner = ScannerAgent(openrouter_client)
        self.messenger = MessagingAgent()
        self._collection = collection
    def run(self, deal: Deal) -> Opportunity:
        est = ensemble_price(deal.product_description)
        return Opportunity(deal=deal, estimate=est, discount=est - deal.price)
    def plan(self, memory=None):
        memory = memory or []
        sel = self.scanner.scan(memory=memory)
        if not sel: return None
        opps = [self.run(d) for d in sel.deals[:2]]
        opps.sort(key=lambda o: o.discount, reverse=True)
        best = opps[0]
        if best.discount > self.DEAL_THRESHOLD:
            self.messenger.alert(best)
        return best if best.discount > self.DEAL_THRESHOLD else None

In [10]:
framework = DealAgentFramework()
framework.planner = PlanningAgentLLM(framework.collection)
opportunities = framework.run()
print(len(opportunities), opportunities[-1] if opportunities else None)

[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent has initialized Pushover and Claude
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent has initialized Pushover and Claude
[2026-03-10 13:18:09 +0100] [Agents] [INFO] Kicking off Planning Agent
[2026-03-10 13:18:09 +0100] [Agents] [INFO] Kicking off Planning Agent
[2026-03-10 13:18:09 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is about to fetch deal

In [15]:
class AutonomousPlannerLLM(Agent):
    name, color = "Autonomous (LLM)", Agent.GREEN
    def __init__(self):
        self.scanner = ScannerAgent(openrouter_client)
        self.messenger = MessagingAgent()
        self.openai = openrouter_client
        self.memory = None
        self.opportunity = None
    def scan_(self):
        r = self.scanner.scan(memory=self.memory)
        return r.model_dump_json() if r else "No deals"
    def estimate_(self, description: str):
        return str(ensemble_price(description))
    def notify_(self, description: str, deal_price: float, est: float, url: str):
        self.messenger.notify(description, deal_price, float(est), url)
        self.opportunity = Opportunity(deal=Deal(product_description=description, price=deal_price, url=url), estimate=float(est), discount=float(est)-deal_price)
        return "ok"
    def get_tools(self):
        return [
            {"type": "function", "function": {"name": "scan", "description": "Get bargains", "parameters": {"type": "object", "properties": {}}}},
            {"type": "function", "function": {"name": "estimate", "description": "Estimate value", "parameters": {"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"]}}},
            {"type": "function", "function": {"name": "notify", "description": "Notify user of deal", "parameters": {"type": "object", "properties": {"description": {"type": "string"}, "deal_price": {"type": "number"}, "estimated_true_value": {"type": "number"}, "url": {"type": "string"}}, "required": ["description", "deal_price", "estimated_true_value", "url"]}}},
        ]
    def handle_tool(self, msg):
        out = []
        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            if name == "scan": res = self.scan_()
            elif name == "estimate": res = self.estimate_(args.get("description", ""))
            elif name == "notify": res = self.notify_(args.get("description", ""), args.get("deal_price", 0), args.get("estimated_true_value", 0), args.get("url", ""))
            else: res = ""
            out.append({"role": "tool", "content": str(res), "tool_call_id": tc.id})
        return out
    def plan(self, memory=None):
        self.memory = memory or []
        self.opportunity = None
        msgs = [{"role": "system", "content": "Find deals, estimate value, notify user of best deal once."}, {"role": "user", "content": "Scan, then estimate each, then notify for the best one. Reply OK when done."}]
        while True:
            r = self.openai.chat.completions.create(model="gpt-4o-mini", messages=msgs, tools=self.get_tools())
            if r.choices[0].finish_reason != "tool_calls": break
            msgs.append(r.choices[0].message)
            msgs.extend(self.handle_tool(r.choices[0].message))
        return self.opportunity

In [16]:
autonomous = AutonomousPlannerLLM()
result = autonomous.plan([])
print(result)

[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent has initialized Pushover and Claude
[2026-03-10 13:22:18 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent has initialized Pushover and Claude
[2026-03-10 13:22:19 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
[2026-03-10 13:22:19 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
[2026-03-10 13

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-120b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'OpenInference', 'is_byok': False}}, 'user_id': 'user_3AJguJZWx8peAUxhSVxYcjb3NwB'}

In [ ]:
app_fw = DealAgentFramework()
app_fw.planner = PlanningAgentLLM(app_fw.collection)

def table(opps):
    return [[o.deal.product_description, o.deal.price, o.estimate, o.discount, o.deal.url] for o in (opps or [])]

with gr.Blocks(title="Price is Right", fill_width=True) as ui:
    state = gr.State(app_fw.memory or [])
    gr.Markdown("**The Price is Right** – OpenAI RAG  + OpenRouter x2")
    df = gr.Dataframe(headers=["Description", "Price", "Estimate", "Discount", "URL"], wrap=True, max_height=400)
    ui.load(table, state, df)
    def run():
        opps = app_fw.run()
        return table(opps), opps
    gr.Button("Run cycle").click(run, outputs=[df, state])
ui.launch(inbrowser=True)

[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is initializing
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is ready
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent is initializing
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent has initialized Pushover and Claude
[2026-03-10 13:25:00 +0100] [Agents] [INFO] [Messaging Agent] Messaging Agent has ini

[2026-03-10 13:25:10 +0100] [Agents] [INFO] Kicking off Planning Agent
[2026-03-10 13:25:10 +0100] [Agents] [INFO] Kicking off Planning Agent
[2026-03-10 13:25:10 +0100] [Agents] [INFO] Kicking off Planning Agent
[2026-03-10 13:25:10 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
[2026-03-10 13:25:10 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
[2026-03-10 13:25:10 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
[2026-03-10 13:26:42 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent received 30 deals not already scraped
[2026-03-10 13:26:42 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent received 30 deals not already scraped
[2026-03-10 13:26:42 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent received 30 deals not already scraped
[2026-03-10 13:26:42 +0100] [Agents] [INFO] [Scanner Agent] Scanner Agent is calling OpenAI using Structured Output

Traceback (most recent call last):
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tim/andela-ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^